# Notebook 6: Explainable AI (SHAP)
## AI-Powered Smart University Assistant

This notebook covers:
- Global SHAP summary plot (feature importance across all students)
- Local SHAP waterfall plot (individual student explanation)
- Feature importance bar chart
- Human-readable explanations for academic advisors

In [ ]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle, sys, os
sys.path.append('..')
warnings = __import__('warnings'); warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
os.makedirs('../screenshots', exist_ok=True)
print('SHAP version:', shap.__version__)

## 1. Load Model and Data

In [ ]:
df = pd.read_csv('../data/processed/student_features.csv')
drop_cols = [c for c in ['id_student', 'code_module', 'code_presentation', 'final_result', 'At_Risk'] if c in df.columns]
X = df.drop(columns=drop_cols)
y = df['At_Risk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

with open('../models/risk_prediction_model.pkl', 'rb') as f:
    pipeline = pickle.load(f)

classifier = pipeline.named_steps['classifier']
preprocessor = pipeline.named_steps['preprocessor']

# Transform the test data
X_test_transformed = preprocessor.transform(X_test)
if hasattr(X_test_transformed, 'toarray'):
    X_test_transformed = X_test_transformed.toarray()

# Get feature names after preprocessing
try:
    feature_names_out = preprocessor.get_feature_names_out()
    feature_names = [n.replace('num__', '').replace('cat__', '').replace('_', ' ').title() for n in feature_names_out]
except:
    feature_names = X.columns.tolist()

print('Data and model loaded.')
print(f'Test set size: {X_test_transformed.shape}')

## 2. Initialize SHAP TreeExplainer

In [ ]:
explainer = shap.TreeExplainer(classifier)
# Use a sample for efficiency
sample_size = min(500, X_test_transformed.shape[0])
X_sample = X_test_transformed[:sample_size]

shap_values = explainer.shap_values(X_sample)

# Handle list output for binary classification
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

print('SHAP values computed.')
print('Shape:', shap_values_class1.shape)

## 3. Global SHAP Summary Plot

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values_class1,
    X_sample,
    feature_names=feature_names[:shap_values_class1.shape[1]],
    show=False,
    plot_type='dot'
)
plt.title('SHAP Global Feature Importance — Academic Risk Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../screenshots/global_shap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Global SHAP plot saved to screenshots/global_shap.png')

## 4. Feature Importance Bar Chart

In [ ]:
mean_abs_shap = np.abs(shap_values_class1).mean(axis=0)
fn = feature_names[:len(mean_abs_shap)]
importance_df = pd.DataFrame({'Feature': fn, 'Importance': mean_abs_shap}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#ef4444' if v > importance_df['Importance'].median() else '#6366f1' for v in importance_df['Importance']]
ax.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
ax.set_title('Mean Absolute SHAP Values — Feature Importance', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.savefig('../screenshots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 5 predictors of academic risk:')
print(importance_df.sort_values('Importance', ascending=False).head(5).to_string(index=False))

## 5. Local SHAP Explanation (Individual Student)

In [ ]:
# Pick a high-risk student (predicted positive)
y_pred_sample = classifier.predict(X_sample)
at_risk_indices = np.where(y_pred_sample == 1)[0]

if len(at_risk_indices) > 0:
    idx = at_risk_indices[0]
    print(f'Explaining prediction for student index: {idx}')
    
    # Get SHAP Explanation object
    shap_explanation = explainer(X_sample[idx:idx+1])
    sv = shap_explanation[0]
    
    # For binary classification
    if len(sv.values.shape) > 1:
        sv.values = sv.values[:, 1]
        sv.base_values = sv.base_values[1]
    
    sv.feature_names = feature_names[:len(sv.values)]
    
    plt.figure(figsize=(10, 5))
    shap.waterfall_plot(sv, show=False)
    plt.title('Local SHAP Explanation — Individual Student', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../screenshots/local_shap.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Human-readable explanation
    sorted_idx = np.argsort(np.abs(sv.values))[::-1]
    top3 = [sv.feature_names[i] for i in sorted_idx[:3]]
    print(f"\nHuman-readable explanation:")
    print(f'This student was classified as AT RISK primarily because of their:')
    for i, feat in enumerate(top3, 1):
        print(f'  {i}. {feat}')

## 6. Discussion — Model Transparency

### Global Explanation
The SHAP summary plot reveals that **Average Assessment Score** is the single strongest predictor of academic risk across all students. Students with scores below 50 are almost always classified as at-risk.

**Total VLE Clicks** and **Active Learning Days** are the next strongest predictors, confirming that student engagement with the learning platform is a leading indicator of academic outcomes.

### Local Explanation
For any individual student, the SHAP waterfall plot shows exactly which factors pushed the model toward or away from the At-Risk classification. Red bars increase risk, blue bars decrease risk.

### Ethical Responsibility
These explanations help academic advisors understand WHY a student was flagged rather than blindly trusting a black-box score. This is critical for:
- Building trust with advisors and students
- Identifying actionable interventions (e.g., 'improve assessment scores')
- Ensuring the model is not using inappropriate features